# Dataset Creation

The goal of this notebook is to create a curated training dataset of protein sequences and secondary structure labels.

## Download Protein Structures

A subset of representative training structures was curating using the [PISCES](https://dunbrack.fccc.edu/pisces/PISCES_OptionPage.php) tool from the Dunbrack lab at FCCC. PDB IDs and Chain IDs from the `selected_protein.txt` are then downloaded from the Protein Data Bank (PDB) using their web API. 

In [1]:
import urllib.request, pathlib, time
from tqdm.auto import tqdm

In [7]:
out = pathlib.Path("Structures")
out.mkdir(exist_ok=True)

def parse_pisces(path):
    ids = []
    with open(path) as f:
        next(f)
        for line in f:
            tok = line.split()[0]
            ids.append((tok[:4].lower(), tok[4:]))  # (pdb_id, chain)
    return ids

entries = parse_pisces("selected_chains.txt")

for pdb_id in tqdm(sorted({p for p, _ in entries}), desc="Structure File Download"):
    dest = out / f"{pdb_id}.cif.gz"
    if dest.exists():
        continue
    part = out / f"{pdb_id}.cif.gz.part"      # download here first so a failure can't leave a corrupt .gz
    try:
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.cif.gz", part)
        part.replace(dest)
    except Exception as e:
        tqdm.write(f"failed {pdb_id}: {e}")
        part.unlink(missing_ok=True)
    time.sleep(0.1)

Structure File Download:   0%|          | 0/8445 [00:00<?, ?it/s]

## Assignment of Secondary Structure

Each structure file (`.cif`) is passed through the standard secondary structure assignment tool [DSSP](https://github.com/PDB-REDO/dssp). This tool is installed in the conda environment.

Each residue is assigned a secondary structure based on electrostatic interactinos, backbone hydrogen-bonding patterns, and torsion angles. It provides 9 structural assignments at each residue position:
- $\alpha$-Helix (H)
- $3_{10}$-Helix (G)
- $\pi$-Helix (I)
- $\beta$-Sheet (E)
- $\beta$-Bridge (B)
- Hydrogen-Bonded Turn (T)
- High Curvature (S)
- Polyproline II Helix (P)
- Coil/Loop (C)

I redefine this to just 3 labels:
- H,G,I &rarr; Helix (H)
- E,B &rarr; Sheet (E)
- T,S,P,C &rarr; Coil (C)

I finally create a dataframe containing the PDB ID, Chain ID, Amino Acid Sequence, and Secondary Structure Label saved to `selected_proteins.csv`.

In [8]:
from collections import defaultdict
from Bio.PDB import MMCIFParser, DSSP
import pandas as pd
import os, sys, shutil, gzip, tempfile
import warnings

def find_first(paths):
    return next((p for p in paths if os.path.exists(p)), None)

exe = "mkdssp.exe" if os.name == "nt" else "mkdssp"
MKDSSP = find_first([
    os.path.join(sys.prefix, "Library", "bin", exe),   # conda on Windows
    os.path.join(sys.prefix, "Scripts", exe),
    os.path.join(sys.prefix, "bin", exe),              # macOS / Linux
]) or shutil.which("mkdssp")
assert MKDSSP, "mkdssp not found in this environment; see the search command below"

DATA_DIR = next(
    (d for d in [
        os.path.join(sys.prefix, "Library", "share", "libcifpp"),   # Windows
        os.path.join(sys.prefix, "share", "libcifpp"),              # macOS / Linux, and your Windows install
    ] if os.path.exists(os.path.join(d, "mmcif_pdbx.dic"))),
    None,
)
assert DATA_DIR, "mmcif_pdbx.dic not found in any expected folder"
os.environ["LIBCIFPP_DATA_DIR"] = DATA_DIR

print("mkdssp:", MKDSSP)
print("data dir:", DATA_DIR)

three_state = {
    **dict.fromkeys("HGI", "H"),
    **dict.fromkeys("EB", "E"),
}  # everything else -> "C"

# Add chain_id to pdb_id for DSSP label
wanted = defaultdict(set)
for pdb_id, chain_id in entries:
    wanted[pdb_id].add(chain_id)

warnings.filterwarnings(
    "ignore",
    message=r".*version in dictionary mmcif_pdbx\.dic is lower than requested.*",
    category=UserWarning,
)

rows = []
for pdb_id, chains in tqdm(wanted.items(), desc="DSSP Processing"):
    path = f"Structures/{pdb_id}.cif.gz"
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            cif = os.path.join(tmpdir, f"{pdb_id}.cif")
            with gzip.open(path, "rb") as fin, open(cif, "wb") as fout:
                shutil.copyfileobj(fin, fout)
            s = MMCIFParser(QUIET=True).get_structure(pdb_id, cif)
            dssp = DSSP(s[0], cif, dssp=MKDSSP, file_type="MMCIF")
    except Exception as e:
        tqdm.write(f"skipping {pdb_id}: {e}")
        continue

    seq, ss = defaultdict(list), defaultdict(list)
    for (ch, _), v in dssp.property_dict.items():
        if ch in chains:
            seq[ch].append(v[1])
            ss[ch].append(three_state.get(v[2], "C"))

    for ch in chains:
        if seq[ch]:
            rows.append({
                "pdb_id": pdb_id,
                "chain": ch,
                "sequence": "".join(seq[ch]),
                "structure": "".join(ss[ch]),
            })

df = pd.DataFrame(rows)
df.to_csv("selected_proteins.csv", index=False)
print(len(df), "chains written")

mkdssp: c:\Users\sjeng\anaconda3\envs\protein_dataset_env\Library\bin\mkdssp.exe
data dir: c:\Users\sjeng\anaconda3\envs\protein_dataset_env\share\libcifpp


DSSP Processing:   0%|          | 0/8445 [00:00<?, ?it/s]

c:\Users\sjeng\anaconda3\envs\protein_dataset_env\lib\python\site-packages\Bio\PDB\DSSP.py:199: UserWarning: The data in this file won't fit in the old DSSP format, please use the mmCIF format instead.

  warnings.warn(err)


skipping 7af2: DSSP failed to produce an output


c:\Users\sjeng\anaconda3\envs\protein_dataset_env\lib\python\site-packages\Bio\PDB\DSSP.py:199: UserWarning: The data in this file won't fit in the old DSSP format, please use the mmCIF format instead.

  warnings.warn(err)


skipping 7pgz: DSSP failed to produce an output
skipping 9qr1: Structure/DSSP mismatch at <Residue CYS het=  resseq=51 icode= >


... (116 outputs left out)

c:\Users\sjeng\anaconda3\envs\protein_dataset_env\lib\python\site-packages\Bio\PDB\DSSP.py:199: UserWarning: The data in this file won't fit in the old DSSP format, please use the mmCIF format instead.

  warnings.warn(err)


skipping 6swz: DSSP failed to produce an output


c:\Users\sjeng\anaconda3\envs\protein_dataset_env\lib\python\site-packages\Bio\PDB\DSSP.py:199: UserWarning: The data in this file won't fit in the old DSSP format, please use the mmCIF format instead.

  warnings.warn(err)


skipping 8oxn: DSSP failed to produce an output
8636 chains written
